In [56]:
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [57]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, ClassVar

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet


In [58]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import pandas as pd
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet


# ============================================================
# 1) FLATTENER : Excel hiérarchique -> table 1 ligne par hôtel
# ============================================================

@dataclass
class HotelExcelFlattener:
    """
    Transforme le fichier Excel ROD hiérarchique en dataframe tabulaire.

    Sortie :
    - 1 ligne = 1 hôtel
    - 1 colonne = 1 variable
    - nom de variable construit uniquement avec les colonnes B, C, D :
        B = étape ROD
        C = sous-étape ROD
        D = data

    Les colonnes E:J ne sont PAS utilisées dans le nom de variable.
    Donc les commentaires ne polluent plus les noms de colonnes.

    Exemple de nom généré :
    etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h
    """

    excel_path: str | Path
    sheet_name: str | None = None

    header_row: int = 3

    # Colonnes utiles pour construire le nom de variable : B, C, D
    variable_name_start_col: int = 2      # B
    variable_name_end_col: int = 4        # D

    # Colonnes métadonnées complètes pour dictionnaire : B:J
    metadata_start_col: int = 2           # B
    metadata_end_col: int = 10            # J

    # Colonnes hôtels : K:Q
    hotel_start_col: int = 11             # K
    hotel_end_col: int = 17               # Q

    separator: str = "__"
    formula_suffix: str = "__formula"
    source_row_suffix: str = "source_row"

    include_hotel_metadata: bool = True
    include_source_row_when_duplicate: bool = True

    _df: pd.DataFrame | None = field(default=None, init=False)
    field_dictionary_: pd.DataFrame | None = field(default=None, init=False)

    def to_df(self) -> pd.DataFrame:
        """
        Retourne la dataframe aplatie.
        """
        if self._df is None:
            self._df = self._build()

        return self._df.copy()

    def _build(self) -> pd.DataFrame:
        excel_path = Path(self.excel_path)

        wb_values = load_workbook(excel_path, data_only=True)
        wb_formulas = load_workbook(excel_path, data_only=False)

        ws_values = self._get_sheet(wb_values)
        ws_formulas = self._get_sheet(wb_formulas)

        metadata_headers = self._read_headers(ws_values)
        hotel_columns = self._detect_hotel_columns(ws_values)

        field_specs = self._build_field_specs(
            ws_values=ws_values,
            ws_formulas=ws_formulas,
            metadata_headers=metadata_headers,
        )

        rows: list[dict[str, Any]] = []

        for hotel in hotel_columns:
            row: dict[str, Any] = {}

            if self.include_hotel_metadata:
                row["hotel__name"] = hotel["name"]
                row["hotel__brand"] = hotel.get("brand")
            else:
                row["hotel"] = hotel["name"]

            for spec in field_specs:
                value = ws_values.cell(
                    row=spec["row_idx"],
                    column=hotel["col_idx"],
                ).value

                row[spec["field_name"]] = value

            rows.append(row)

        self.field_dictionary_ = pd.DataFrame(field_specs)

        return pd.DataFrame(rows)

    def _get_sheet(self, workbook) -> Worksheet:
        return workbook[self.sheet_name] if self.sheet_name else workbook[workbook.sheetnames[0]]

    def _read_headers(self, ws: Worksheet) -> dict[int, str]:
        """
        Lit les intitulés des colonnes B:J.
        """
        headers: dict[int, str] = {}

        for col_idx in range(self.metadata_start_col, self.metadata_end_col + 1):
            raw = ws.cell(row=self.header_row, column=col_idx).value

            if self._has_value(raw):
                headers[col_idx] = self._clean_text(raw)
            else:
                headers[col_idx] = f"metadata_col_{col_idx}"

        return headers

    def _detect_hotel_columns(self, ws: Worksheet) -> list[dict[str, Any]]:
        """
        Détecte les colonnes hôtels K:Q.

        Ligne 2 : marque, parfois fusionnée.
        Ligne 3 : nom hôtel.
        """
        hotels: list[dict[str, Any]] = []
        current_brand: str | None = None

        for col_idx in range(self.hotel_start_col, self.hotel_end_col + 1):
            raw_brand = ws.cell(row=self.header_row - 1, column=col_idx).value

            if self._has_value(raw_brand):
                current_brand = str(raw_brand).strip()

            raw_name = ws.cell(row=self.header_row, column=col_idx).value

            if not self._has_value(raw_name):
                continue

            hotels.append(
                {
                    "col_idx": col_idx,
                    "name": str(raw_name).strip(),
                    "brand": current_brand,
                }
            )

        if not hotels:
            raise ValueError(
                "Aucune colonne hôtel détectée. "
                "Vérifie hotel_start_col, hotel_end_col et header_row."
            )

        return hotels

    def _build_field_specs(
        self,
        *,
        ws_values: Worksheet,
        ws_formulas: Worksheet,
        metadata_headers: dict[int, str],
    ) -> list[dict[str, Any]]:
        """
        Construit les variables à partir de B, C, D uniquement.

        B/C peuvent être fusionnées ou vides sur certaines lignes :
        on applique donc un forward-fill pour garder la hiérarchie.
        """
        specs: list[dict[str, Any]] = []

        forward_fill: dict[int, Any] = {}

        for row_idx in range(self.header_row + 1, ws_values.max_row + 1):

            # 1. Mise à jour du contexte hiérarchique B:J
            for col_idx in range(self.metadata_start_col, self.metadata_end_col + 1):
                raw_value = ws_values.cell(row=row_idx, column=col_idx).value

                if self._has_value(raw_value):
                    forward_fill[col_idx] = raw_value

            # 2. Construction du nom de variable uniquement avec B, C, D
            variable_parts: list[str] = []

            for col_idx in range(self.variable_name_start_col, self.variable_name_end_col + 1):
                value = forward_fill.get(col_idx)

                if not self._has_value(value):
                    continue

                header = metadata_headers[col_idx]
                variable_parts.extend(
                    [
                        header,
                        self._clean_text(value),
                    ]
                )

            if not variable_parts:
                continue

            base_field_name = self.separator.join(variable_parts)

            if self._row_contains_formula(ws_formulas, row_idx):
                base_field_name = f"{base_field_name}{self.formula_suffix}"

            # 3. Métadonnées complètes pour audit uniquement, pas pour nommage
            full_metadata = {}

            for col_idx in range(self.metadata_start_col, self.metadata_end_col + 1):
                header = metadata_headers[col_idx]
                value = forward_fill.get(col_idx)
                full_metadata[header] = value

            specs.append(
                {
                    "row_idx": row_idx,
                    "field_name": base_field_name,
                    "variable_name_bcd": base_field_name,
                    "source_metadata": full_metadata,
                    "is_formula": self._row_contains_formula(ws_formulas, row_idx),
                }
            )

        if self.include_source_row_when_duplicate:
            specs = self._deduplicate_field_names(specs)

        return specs

    def _row_contains_formula(self, ws: Worksheet, row_idx: int) -> bool:
        """
        Retourne True si au moins une cellule hôtel de la ligne contient une formule Excel.
        """
        for col_idx in range(self.hotel_start_col, self.hotel_end_col + 1):
            value = ws.cell(row=row_idx, column=col_idx).value

            if isinstance(value, str) and value.startswith("="):
                return True

        return False

    def _deduplicate_field_names(self, specs: list[dict[str, Any]]) -> list[dict[str, Any]]:
        """
        Si deux lignes produisent le même nom B/C/D,
        on ajoute source_row pour éviter l'écrasement.
        """
        counts = Counter(spec["field_name"] for spec in specs)

        for spec in specs:
            if counts[spec["field_name"]] > 1:
                spec["field_name"] = (
                    f"{spec['field_name']}"
                    f"{self.separator}{self.source_row_suffix}"
                    f"{self.separator}{spec['row_idx']}"
                )

        return specs

    @staticmethod
    def _has_value(value: Any) -> bool:
        return value is not None and str(value).strip() != ""

    @staticmethod
    def _clean_text(value: Any) -> str:
        """
        Nettoie une valeur pour nom de colonne :
        - minuscules
        - accents retirés
        - espaces remplacés par _
        - ponctuation supprimée
        """
        text = str(value).strip().lower().replace("\n", " ")

        text = unicodedata.normalize("NFKD", text)
        text = "".join(
            char for char in text
            if not unicodedata.combining(char)
        )

        text = re.sub(r"\s+", "_", text)
        text = re.sub(r"[^a-zA-Z0-9_]+", "_", text)
        text = re.sub(r"_+", "_", text)

        return text.strip("_")

In [59]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import ClassVar

import pandas as pd


@dataclass
class HotelDataPrep:
    """
    Prépare la dataframe aplatie pour le ML.

    Règle stricte :
    - max 5 colonnes protégées métier :
        code_h
        nom_de_l_hotel
        adresse_postale_1
        latitude
        longitude

    Aucune protection large par mots-clés.
    """

    df_flat: pd.DataFrame

    protected_fields: tuple[str, ...] = (
        "code_h",
        "nom_de_l_hotel",
        "adresse_postale_1",
        "latitude",
        "longitude",
    )

    remove_patterns: tuple[str, ...] = (
        "simulateur_de_revenus",
        "fictive",
    )

    remove_formula_columns: bool = True
    drop_constant_columns: bool = True
    convert_boolean_like_columns: bool = True
    convert_numeric_like_columns: bool = True
    missing_as_false_for_boolean: bool = True

    _df_ml: pd.DataFrame | None = field(default=None, init=False)

    report_: dict[str, list[str]] = field(default_factory=dict, init=False)

    TRUE_VALUES: ClassVar[set[str]] = {
        "X", "OUI", "YES", "TRUE", "VRAI", "1"
    }

    FALSE_VALUES: ClassVar[set[str]] = {
        "-", "NON", "NO", "FALSE", "FAUX", "0"
    }

    MISSING_VALUES: ClassVar[set[str]] = {
        "", "?", "NA", "N/A", "NULL", "NONE", "NAN", "<NA>"
    }

    def to_df(self) -> pd.DataFrame:
        if self._df_ml is None:
            self._df_ml = self._build()
        return self._df_ml.copy()

    def _build(self) -> pd.DataFrame:
        df = self.df_flat.copy()

        self.report_ = {
            "protected_columns": [],
            "boolean_columns": [],
            "numeric_columns": [],
            "removed_all_missing": [],
            "removed_pattern": [],
            "removed_formula": [],
            "removed_constant": [],
            "removed_ml_strict": [],
        }

        df = self._standardize_missing_values(df)

        protected_cols = self._select_exact_protected_columns(df)
        self.report_["protected_columns"] = protected_cols

        if self.convert_boolean_like_columns:
            df = self._convert_boolean_columns(df)

        if self.convert_numeric_like_columns:
            df = self._convert_numeric_columns(df)

        remove_cols: set[str] = set()

        all_missing_cols = [
            c for c in df.columns
            if c not in protected_cols
            and self._is_all_missing(df[c])
        ]
        remove_cols.update(all_missing_cols)
        self.report_["removed_all_missing"] = all_missing_cols

        pattern_cols = [
            c for c in df.columns
            if c not in protected_cols
            and c not in remove_cols
            and self._contains_any(c, self.remove_patterns)
        ]
        remove_cols.update(pattern_cols)
        self.report_["removed_pattern"] = pattern_cols

        if self.remove_formula_columns:
            formula_cols = [
                c for c in df.columns
                if c not in protected_cols
                and c not in remove_cols
                and (c.endswith("__formula") or c.endswith("formula"))
            ]
            remove_cols.update(formula_cols)
            self.report_["removed_formula"] = formula_cols

        if self.drop_constant_columns:
            constant_cols = [
                c for c in df.columns
                if c not in protected_cols
                and c not in remove_cols
                and self._is_constant(df[c])
            ]
            remove_cols.update(constant_cols)
            self.report_["removed_constant"] = constant_cols

        keep_cols = [
            c for c in df.columns
            if c not in remove_cols
        ]

        return df[keep_cols].copy()

    def _select_exact_protected_columns(self, df: pd.DataFrame) -> list[str]:
        """
        Sélectionne exactement une colonne source par champ métier.
        Max = len(self.protected_fields), donc 5 colonnes max.

        On ne protège PAS :
        - hotel__name
        - hotel__brand
        - adresse_postale_2
        - adresse_postale_3
        - ville
        - code_postal
        - hotel_code
        - code_hotel
        """

        selected: list[str] = []

        for wanted_field in self.protected_fields:
            candidates = [
                col for col in df.columns
                if self._extract_data_field(col) == wanted_field
            ]

            if not candidates:
                continue

            best_col = self._choose_best_column(candidates, df)
            selected.append(best_col)

        if len(selected) > 5:
            raise ValueError(
                f"Trop de colonnes protégées détectées : {len(selected)}. "
                f"Colonnes : {selected}"
            )

        return selected

    @staticmethod
    def _extract_data_field(column_name: str) -> str | None:
        """
        Extrait uniquement le champ après '__data__'.

        Exemple :
        ...__data__code_h__deja_dans_rod...
        retourne :
        code_h
        """

        marker = "__data__"

        if marker not in column_name:
            return None

        after = column_name.split(marker, 1)[1]

        if "__" not in after:
            return after

        return after.split("__", 1)[0]

    @staticmethod
    def _choose_best_column(candidates: list[str], df: pd.DataFrame) -> str:
        """
        S'il y a plusieurs colonnes pour le même champ,
        on garde celle qui contient le plus de valeurs non vides.
        """

        return max(
            candidates,
            key=lambda col: df[col].notna().sum()
        )

    def _standardize_missing_values(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        for col in df.columns:
            if df[col].dtype == "object":
                s = df[col].astype("string").str.strip()
                s = s.mask(s.str.upper().isin(self.MISSING_VALUES), pd.NA)
                df[col] = s

        return df

    def _convert_boolean_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                continue

            s = df[col].astype("string").str.strip().str.upper()
            unique_values = set(s.dropna().unique())

            allowed = self.TRUE_VALUES | self.FALSE_VALUES

            if unique_values and unique_values <= allowed:
                mapped = s.map(
                    {
                        **{v: 1 for v in self.TRUE_VALUES},
                        **{v: 0 for v in self.FALSE_VALUES},
                    }
                )

                if self.missing_as_false_for_boolean:
                    mapped = mapped.fillna(0)

                df[col] = mapped.astype("Int8")
                self.report_["boolean_columns"].append(col)

        return df

    def _convert_numeric_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                self.report_["numeric_columns"].append(col)
                continue

            converted = self._try_parse_numeric(df[col])

            if converted is not None:
                df[col] = converted
                self.report_["numeric_columns"].append(col)

        return df

    @staticmethod
    def _try_parse_numeric(s: pd.Series) -> pd.Series | None:
        raw = s.astype("string").str.strip()

        cleaned = (
            raw
            .str.replace("\u202f", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.replace("€", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        numeric = pd.to_numeric(cleaned, errors="coerce")

        original_non_null = raw.dropna().shape[0]
        numeric_non_null = numeric.dropna().shape[0]

        if original_non_null == 0:
            return None

        if numeric_non_null / original_non_null >= 0.8:
            return numeric

        return None

    @staticmethod
    def _is_all_missing(s: pd.Series) -> bool:
        s_clean = s.replace(
            ["", " ", "?", "<NA>", "nan", "NaN", "None"],
            pd.NA
        )
        return s_clean.isna().all()

    @staticmethod
    def _is_constant(s: pd.Series) -> bool:
        s_clean = s.replace(
            ["", " ", "?", "<NA>", "nan", "NaN", "None"],
            pd.NA
        )
        return s_clean.dropna().nunique() <= 1

    @staticmethod
    def _contains_any(text: str, patterns: tuple[str, ...]) -> bool:
        text = text.lower()
        return any(pattern.lower() in text for pattern in patterns)
    

    def to_ml_strict_df(self) -> pd.DataFrame:
        """
        Version stricte du dataset ML :
        supprime toutes les colonnes contenant au moins une valeur manquante
        (hors colonnes protégées).
        """

        df = self.to_df().copy()

        protected_cols = self.report_["protected_columns"]

        strict_removed = [
            col for col in df.columns
            if col not in protected_cols
            and df[col].isna().any()
        ]

        df = df.drop(columns=strict_removed)

        self.report_["removed_ml_strict"] = strict_removed

        return df

In [62]:
# ============================================================
# 3) USAGE AVEC AUDIT DE L'ÉVOLUTION DES COLONNES
# ============================================================

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

excel_path = "Récapitulatif de l'ensemble des données ROD enrichies.xlsx"

# 1. Flatten
flattener = HotelExcelFlattener(excel_path)
df_flat = flattener.to_df()

print("=" * 80)
print("1. DATAFRAME FLATTENED")
print("=" * 80)
print("Shape flattened:", df_flat.shape)
display(df_flat)

# 2. Data prep ML
prep = HotelDataPrep(df_flat)
df_ml = prep.to_df()

print("=" * 80)
print("2. DATAFRAME ML READY")
print("=" * 80)
print("Shape ML:", df_ml.shape)
display(df_ml)


# ML strict
df_ml_strict = prep.to_ml_strict_df()

print("=" * 80)
print("3. DATAFRAME ML STRICT")
print("=" * 80)
print("Shape ML strict:", df_ml_strict.shape)
display(df_ml_strict)

# 3. Audit synthétique
audit_rows = []

n_start = df_flat.shape[1]

protected_cols = prep.report_["protected_columns"]
boolean_cols = prep.report_["boolean_columns"]
numeric_cols = prep.report_["numeric_columns"]
removed_all_missing = prep.report_["removed_all_missing"]
removed_pattern = prep.report_["removed_pattern"]
removed_formula = prep.report_["removed_formula"]
removed_constant = prep.report_["removed_constant"]
removed_ml_strict = prep.report_["removed_ml_strict"]

removed_total = (
    len(removed_all_missing)
    + len(removed_pattern)
    + len(removed_formula)
    + len(removed_constant)
)

audit_rows.append(
    {
        "phase": "Départ - df_flat",
        "nb_colonnes": n_start,
        "delta": 0,
        "commentaire": "Colonnes issues du fichier Excel aplati",
    }
)

audit_rows.append(
    {
        "phase": "Colonnes protégées",
        "nb_colonnes": n_start,
        "delta": 0,
        "commentaire": f"{len(protected_cols)} colonnes protégées max attendues = 5",
    }
)

audit_rows.append(
    {
        "phase": "Conversion booléenne",
        "nb_colonnes": n_start,
        "delta": 0,
        "commentaire": f"{len(boolean_cols)} colonnes converties en 0/1",
    }
)

audit_rows.append(
    {
        "phase": "Conversion numérique",
        "nb_colonnes": n_start,
        "delta": 0,
        "commentaire": f"{len(numeric_cols)} colonnes numériques détectées/converties",
    }
)

n_after_missing = n_start - len(removed_all_missing)
audit_rows.append(
    {
        "phase": "Suppression colonnes vides",
        "nb_colonnes": n_after_missing,
        "delta": -len(removed_all_missing),
        "commentaire": f"{len(removed_all_missing)} colonnes 100% vides supprimées",
    }
)

n_after_pattern = n_after_missing - len(removed_pattern)
audit_rows.append(
    {
        "phase": "Suppression simulateur/fictive",
        "nb_colonnes": n_after_pattern,
        "delta": -len(removed_pattern),
        "commentaire": f"{len(removed_pattern)} colonnes supprimées par pattern métier",
    }
)

n_after_formula = n_after_pattern - len(removed_formula)
audit_rows.append(
    {
        "phase": "Suppression formules",
        "nb_colonnes": n_after_formula,
        "delta": -len(removed_formula),
        "commentaire": f"{len(removed_formula)} colonnes formule supprimées",
    }
)

n_after_constant = n_after_formula - len(removed_constant)
audit_rows.append(
    {
        "phase": "Suppression constantes",
        "nb_colonnes": n_after_constant,
        "delta": -len(removed_constant),
        "commentaire": f"{len(removed_constant)} colonnes constantes supprimées",
    }
)

audit_rows.append(
    {
        "phase": "Final - df_ml",
        "nb_colonnes": df_ml.shape[1],
        "delta": df_ml.shape[1] - n_start,
        "commentaire": "Dataset final prêt pour jointure / ML",
    }
)

audit_rows.append(
    {
        "phase": "ML strict (remove partial missing)",
        "nb_colonnes": df_ml_strict.shape[1],
        "delta": -len(removed_ml_strict),
        "commentaire": f"{len(removed_ml_strict)} colonnes avec NA supprimées",
    }
)

df_audit = pd.DataFrame(audit_rows)

print("=" * 80)
print("3. AUDIT ÉVOLUTION DES COLONNES")
print("=" * 80)
display(df_audit)

# 4. Détails utiles
print("=" * 80)
print("4. COLONNES PROTÉGÉES")
print("=" * 80)
print(f"Nombre de colonnes protégées : {len(protected_cols)}")
for col in protected_cols:
    print("-", col)

print("=" * 80)
print("5. CONTRÔLE QUALITÉ")
print("=" * 80)

if len(protected_cols) > 5:
    raise ValueError(
        f"Erreur : trop de colonnes protégées ({len(protected_cols)}). "
        "Il faut max 5."
    )

if df_ml.isna().all().any():
    empty_cols = df_ml.columns[df_ml.isna().all()].tolist()
    print("Attention : colonnes encore totalement vides :")
    for col in empty_cols:
        print("-", col)
else:
    print("OK : aucune colonne totalement vide dans df_ml.")

constant_cols_remaining = [
    c for c in df_ml.columns
    if c not in protected_cols
    and df_ml[c].dropna().nunique() <= 1
]

if constant_cols_remaining:
    print("Attention : colonnes constantes restantes :")
    for col in constant_cols_remaining:
        print("-", col)
else:
    print("OK : aucune colonne constante non protégée dans df_ml.")

# 6. Export
df_flat.to_excel("rod_hotels_flattened.xlsx", index=False)
df_ml.to_excel("rod_hotels_ml_ready.xlsx", index=False)
df_audit.to_excel("rod_hotels_audit_dataprep.xlsx", index=False)
df_ml_strict.to_excel("rod_prepared_data.xlsx", index = False)

print("=" * 80)
print("6. EXPORTS TERMINÉS")
print("=" * 80)
print("rod_hotels_flattened.xlsx")
print("rod_hotels_ml_ready.xlsx")
print("rod_hotels_audit_dataprep.xlsx")

1. DATAFRAME FLATTENED
Shape flattened: (7, 144)


,hotel__name,hotel__brand,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_3,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__nb_de_chambres,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_signe_annee,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_type,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__proprietaire,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__dom_dof,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_hotel,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_lobby,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__pms,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__adultes_par_chambre,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__enfants_par_chambre,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__panier_moyen,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_annuel,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_bas_mois,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_bas_taux,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_haut_mois,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_haut_taux,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__a_proximite_a_100m,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_restauration__source_row__31,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_supermarches__source_row__32,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_autres_commerces__source_row__33,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__a_proximite_a_500m,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_restauration__source_row__35,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_supermarches__source_row__36,etape_rod__1_informations_generales__sous_etape_rod__localisation_environnement__data__nb_autres_commerces__source_row__37,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__bar,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__horaires_d_ouverture__source_row__39,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__jours_d_ouverture__source_row__40,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__mois_d_ouverture__source_row__41,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__restaurant,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__horaires_d_ouverture__source_row__43,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__jours_d_ouverture__source_row__44,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__mois_d_ouverture__source_row__45,etape_rod__2_services_equipements

2. DATAFRAME ML READY
Shape ML: (7, 71)


,hotel__name,hotel__brand,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__nb_de_chambres,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_signe_annee,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_type,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__proprietaire,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__dom_dof,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_hotel,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_lobby,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__pms,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_annuel,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_bas_taux,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_haut_mois,etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__to_le_plus_haut_taux,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__bar,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__horaires_d_ouverture__source_row__39,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__jours_d_ouverture__source_row__40,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__restaurant,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__horaires_d_ouverture__source_row__43,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__jours_d_ouverture__source_row__44,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__mois_d_ouverture__source_row__45,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__minibar,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salles_de_reunion,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salle_de_sport,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__piscine,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__vitrine_refrigeree,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__micro_ondes,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__fontaine_a_eau,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__machine_a_cafe,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__bouilloire,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__loisirs,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_couples,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_amis,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_familles,etape_rod__3_profil_de_vos_clients__sous_etape_rod__affaires__data__affaires,etape_rod__3_profil_de_vos_clients__sous_etape_rod__national_vs_inter__data__national,etape_rod__3_profil_de_vos_clients__sous_etape_rod__national_vs_inter__data__international,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__boissons_alcoolisees,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__pro

3. DATAFRAME ML STRICT
Shape ML strict: (7, 53)


,hotel__name,hotel__brand,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__nb_de_chambres,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_type,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__proprietaire,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__dom_dof,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__pms,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__bar,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__restaurant,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__minibar,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salles_de_reunion,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salle_de_sport,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__piscine,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__vitrine_refrigeree,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__micro_ondes,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__fontaine_a_eau,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__machine_a_cafe,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__bouilloire,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_couples,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_amis,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_familles,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__boissons_alcoolisees,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__produits_sales_frais,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__produits_sucres_frais,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__epicerie_fine,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__produits_cosmetiques,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__articles_pour_enfants,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__pret_a_porter,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__accessoires,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__souvenirs,etape_rod__4_informations_corner__sous_etape_rod__corner_de_vente_actuel__data__votre_hotel_dispose_t_il_deja_d_un_corner_de_vente,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__liste_des_produits_f_b__source_row__85,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__reception,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__snacking_comptoir,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__distributeur_auto,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__hotel_staff,etape_rod__4_informations_corner__sous_etape_rod__votre_corner

3. AUDIT ÉVOLUTION DES COLONNES


,phase,nb_colonnes,delta,commentaire
0,Départ - df_flat,144,0,Colonnes issues du fichier Excel aplati
1,Colonnes protégées,144,0,5 colonnes protégées max attendues = 5
2,Conversion booléenne,144,0,45 colonnes converties en 0/1
3,Conversion numérique,144,0,74 colonnes numériques détectées/converties
4,Suppression colonnes vides,94,-50,50 colonnes 100% vides supprimées
5,Suppression simulateur/fictive,90,-4,4 colonnes supprimées par pattern métier
6,Suppression formules,90,0,0 colonnes formule supprimées
7,Suppression constantes,71,-19,19 colonnes constantes supprimées
8,Final - df_ml,71,-73,Dataset final prêt pour jointure / ML
9,ML strict (remove partial missing),53,-18,18 colonnes avec NA supprimées


4. COLONNES PROTÉGÉES
Nombre de colonnes protégées : 5
- etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h
- etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel
- etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1
- etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude
- etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude
5. CONTRÔLE QUALITÉ
OK : aucune colonne totalement vide dans df_ml.
OK : aucune colonne constante non protégée dans df_ml.
6. EXPORTS TERMINÉS
rod_hotels_flattened.xlsx
rod_hotels_ml_ready.xlsx
rod_hotels_audit_dataprep.xlsx


In [63]:
df_ml_strict

,hotel__name,hotel__brand,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__nb_de_chambres,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_type,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__proprietaire,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__dom_dof,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__pms,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__bar,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__restaurant,etape_rod__2_services_equipements__sous_etape_rod__f_b__data__minibar,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salles_de_reunion,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salle_de_sport,etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__piscine,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__vitrine_refrigeree,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__micro_ondes,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__fontaine_a_eau,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__machine_a_cafe,etape_rod__2_services_equipements__sous_etape_rod__dispo_dans_le_lobby__data__bouilloire,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_couples,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_amis,etape_rod__3_profil_de_vos_clients__sous_etape_rod__loisirs__data__top_1_familles,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__boissons_alcoolisees,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__produits_sales_frais,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__produits_sucres_frais,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_f_b__data__epicerie_fine,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__produits_cosmetiques,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__articles_pour_enfants,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__pret_a_porter,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__accessoires,etape_rod__3_profil_de_vos_clients__sous_etape_rod__besoins_de_vos_clients_non_f_b__data__souvenirs,etape_rod__4_informations_corner__sous_etape_rod__corner_de_vente_actuel__data__votre_hotel_dispose_t_il_deja_d_un_corner_de_vente,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__liste_des_produits_f_b__source_row__85,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__reception,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__snacking_comptoir,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__distributeur_auto,etape_rod__4_informations_corner__sous_etape_rod__votre_corner_actuel_offre_f_b__data__hotel_staff,etape_rod__4_informations_corner__sous_etape_rod__votre_corner

In [64]:
df_ml_strict.transpose()

,0,1,2,3,4,5,6
hotel__name,NICE,STRASBOURG,PARIS CDG,MEGEVE,TOUR EIFFEL,MONTMARTRE,BOULOGNE
hotel__brand,IBIS BUDGET,IBIS BUDGET,IBIS STYLES,NOVOTEL,NOVOTEL,MERCURE,MERCURE
etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h,H2075,HB6A3,H0815,HB5I0,H3546,H0373,H6188
etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel,IBIS BUDGET NICE CALIFORNIE,IBIS BUDGET STRASBOURG REPUBLIQUE,IBIS STYLES ROISSY CDG,NOVOTEL MEGEVE MONT BLANC,NOVOTEL PARIS CENTRE TOUR EIFFEL,MERCURE MONTMARTRE SACRE COEUR,MERCURE PARIS BOULOGNE
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1,58-60 AVENUE DE LA CALIFORNIE,23A RUE OBERLIN,2 AVENUE HEINZ GLOOR,1306 ROUTE NATIONALE,61 QUAI DE GRENELLE,3 RUE CAULAINCOURT,37 PLACE RENÉ CLAIR
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2,-,-,-,LE DOMAINE DE MEZTIVA,-,-,-
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal,6200,67000,95700,74120,75015,75018,92100
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville,NICE,STRASBOURG,ROISSY,MEGEVE,PARIS,PARIS,BOULOGNE-BILLANCOURT
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude,7.240512,7.754599,2.519843,6.619055,2.282836,2.329923,2.256274
etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude,43.689186,48.591522,49.006733,45.859165,48.849778,48.885048,48.833827
